# Detección de lavado de dinero sobre datos sintéticos propios (generados en GCP, en shards)

**Tercera versión independiente** de la misma pregunta que `Fraud_Detection.ipynb` y
`Fraud_Detection_AML_HF.ipynb`: clustering, clasificación y detección de anomalías por
reconstrucción — ahora sobre el dataset generado por
[`Synthetic_Data`](https://github.com/ChemaSarmiento/Synthetic_Data), el motor propio de
datos sintéticos (dominio *banking*, escenario opcional *AML*) que se está corriendo en
Cloud Run Jobs y subiendo a Cloud Storage **en shards independientes**.

## Por qué este dataset es distinto a los otros dos

- **`bank_transactions.csv`**: una tabla plana, ya fija, sin proceso de generación visible.
- **AML de IBM (`Fraud_Detection_AML_HF`)**: un benchmark de investigación de terceros, con
  esquema fijo y sin trazabilidad de cómo se generó.
- **Este dataset**: **tú controlas el generador** — el motor documenta exactamente qué
  distribuciones usa, qué patrones de lavado inyecta (`fan_in`, `fan_out`, `chain`, `cycle`),
  y separa explícitamente el label (`ground_truth/`) de las transacciones. Es el único de
  los tres con **features de historial ya precalculadas** por cuenta
  (`sender_prior_out_count`, `sender_prior_mean_usd`, etc., calculadas *solo* con eventos
  anteriores — sin fuga de futuro) y con **controles benignos** que imitan la topología de
  los casos de lavado (`legitimate_motif_fraction`) para que el modelo no aprenda
  "topología de 4 transacciones en 90 min = lavado" como atajo.

## Tres reglas de lectura (documentadas en el propio repo, no opcionales)

1. **Leer `dataset.json` primero.** Solo se publica cuando el run completo fue validado y
   subido — es la única fuente confiable de qué shards existen. Nunca escanear el bucket a
   ciegas.
2. **`transactions/` y `ground_truth/` son tablas separadas.** El label (`is_laundering`)
   no vive en la tabla de transacciones — hay que unirlas por `(shard_id, transaction_id)`.
3. **Sin continuidad entre shards.** Cada uno es una población independiente (cuentas,
   bancos, historial). Cualquier split de train/test debe respetar `shard_id`.

**Importante:** `scenario_motif` (en `ground_truth/`) **no implica lavado** — hay motivos
benignos con la misma topología, a propósito. El label correcto es siempre
`is_laundering`.

## Cluster: sección B de MLOPS/README.md alcanza

A diferencia de `Fraud_Detection_AML_HF` (que necesita `huggingface_hub`/B.1), este
notebook solo lee Parquet de Cloud Storage — el cluster normal de la sección B es
suficiente.

## Parámetros — completar antes de correr

Los shards se generan con `configs/gcp/aml-pilot.json` del motor (preset de literatura,
**no** el perfil calibrado con IBM que sí usa `Fraud_Detection_AML_HF`) — no mezclar
ambos supuestos al comparar resultados entre notebooks.

In [ ]:
# Parche de compatibilidad para el Jupyter de Dataproc: matplotlib llama a RcParams._get,
# que no existe en este entorno. Debe ejecutarse ANTES de importar pyplot.
import matplotlib
_RcParams = type(matplotlib.rcParams)
if not hasattr(_RcParams, '_get'):
    _RcParams._get = _RcParams.get

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, PCA
from pyspark.ml.functions import vector_to_array
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator

In [ ]:
spark = SparkSession.builder.appName("AML_Synthetic_GCP").getOrCreate()

In [ ]:
# --- Completar con los valores reales una vez que la generación termine ---
BUCKET = "big-data-lunes-20260223"     # bucket donde corrió el Cloud Run Job
EXECUTION = "<EXECUTION>"              # el namespace real (CLOUD_RUN_EXECUTION del job)

ROOT = f"gs://{BUCKET}/datasets/{EXECUTION}"

## 1. Leer `dataset.json`: ¿el dataset está completo?

Antes de gastar cluster en leer Parquet, confirmamos que el run terminó y cuántos shards
hay. Si `dataset.json` no existe todavía, la generación sigue corriendo — no hay dataset
"parcial" válido que leer (el runner solo lo publica al final).

In [ ]:
manifest_txt = spark.read.text(f"{ROOT}/dataset.json").collect()
manifest = json.loads("".join(r.value for r in manifest_txt))

print(f"Estado: {manifest['status']}")
print(f"Shards: {len(manifest['shards'])}  |  Filas totales: {manifest['rows']:,}")
print(f"Bytes de transacciones: {manifest.get('transaction_bytes', 0):,}")
assert manifest["status"] == "complete", "El dataset no terminó de generarse — no seguir leyendo"

## 2. Leer transacciones y ground truth, unir por `(shard_id, transaction_id)`

`spark.read.parquet("ROOT/shard_id=*/transactions")` con el comodín **no funciona aquí**:
probado en local, Spark tira `[CONFLICTING_DIRECTORY_STRUCTURES]` porque cada shard ya
resuelve a una ruta concreta distinta y Spark no tiene un único patrón de partición que
inferir entre ellas — ni siquiera con `basePath`. El propio mensaje de error de Spark lo
dice: *"si hay varias raíces, cárgalas por separado y únelas"*. Es exactamente para esto
que leímos `dataset.json` arriba: iteramos los shards que el manifiesto dice que existen,
leemos cada uno por separado (ahí sí se infiere `event_date` sin ambigüedad, porque dentro
de un shard solo hay un nivel de partición) y agregamos `shard_id` como columna explícita
a partir del manifiesto — no confiamos en que Spark la infiera sola de la ruta.

In [ ]:
from functools import reduce

def leer_shards(subcarpeta):
    partes = [spark.read.parquet(f"{ROOT}/shard_id={sh['shard_id']}/{subcarpeta}")
                   .withColumn("shard_id", F.lit(sh['shard_id']))
              for sh in manifest["shards"]]
    return reduce(lambda a, b: a.unionByName(b), partes)

tx = leer_shards("transactions")
gt = leer_shards("ground_truth")

tx.printSchema()

In [ ]:
df_raw = tx.join(gt.select("shard_id", "transaction_id", "is_laundering", "scenario_motif"),
                 on=["shard_id", "transaction_id"], how="inner")

n_total = df_raw.count()
n_laundering = df_raw.filter(col("is_laundering") == True).count()
print(f"Transacciones: {n_total:,} | lavado: {n_laundering:,} ({n_laundering / n_total:.3%})")
print(f"Shards presentes: {[r.shard_id for r in df_raw.select('shard_id').distinct().collect()]}")

print("\nMotivo vs. label real — confirma que scenario_motif != none NO implica lavado:")
df_raw.groupBy("scenario_motif").agg(
    F.count("*").alias("transacciones"),
    F.round(F.avg(col("is_laundering").cast("double")) * 100, 2).alias("pct_lavado")
).orderBy("scenario_motif").show()

## 3. Features

`model_features.json` del propio motor excluye los identificadores
(`transaction_id`, `timestamp`, `sender_account_id`, `receiver_account_id`,
`sender_bank_id`, `receiver_bank_id`) y, en el esquema de pagos v2, tres columnas que
fugan la fuente del dato (`source_payment_format`, `payment_format_origin`,
`payment_mapping_status`) — los excluimos aquí también.

Elegimos un subconjunto manejable, con dos features que **no existen** en los otros dos
notebooks: `sender_prior_out_count` y `sender_prior_mean_usd` (historial de la cuenta,
precalculado solo con eventos anteriores).

In [ ]:
df_base = (df_raw
    .withColumn("amount", col("amount_usd_equivalent"))
    .withColumn("hour", col("sender_local_hour").cast("int"))
    .withColumn("day", col("utc_weekday").cast("int"))
    .withColumn("cruce_pais", col("is_cross_border").cast("string"))
    .withColumn("mismo_dueño", col("same_owner").cast("string"))
    .withColumn("label_real", col("is_laundering").cast("double"))
    .select("amount", "hour", "day", "payment_rail", "purpose", "sender_bank_type",
            "cruce_pais", "mismo_dueño", "sender_prior_out_count", "sender_prior_mean_usd",
            "label_real")
    .na.drop())

df_base.show(5)

# BLOQUE A — Clustering (segmentación de riesgo)

Mismo patrón que en los otros dos notebooks: sin usar la etiqueta, agrupar transacciones
parecidas y ver si el lavado se concentra en algún grupo.

In [ ]:
cat_cols = ["payment_rail", "purpose", "sender_bank_type", "cruce_pais", "mismo_dueño"]
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="skip") for c in cat_cols]
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols], outputCols=[c + "_vec" for c in cat_cols])
assembler = VectorAssembler(
    inputCols=["amount", "hour", "day", "sender_prior_out_count", "sender_prior_mean_usd"]
              + [c + "_vec" for c in cat_cols],
    outputCol="features_raw"
)
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled")

In [ ]:
%%time
prep_pipeline = Pipeline(stages=indexers + [encoder, assembler, scaler])
prep_model = prep_pipeline.fit(df_base)
df_final = prep_model.transform(df_base)
df_final.select("amount", "hour", "day", "label_real", "features_scaled").show(5, truncate=60)

### ¿Cuántos grupos? El método del codo

In [ ]:
%%time
K_MAX = 6
cost = []
print(f"Calculando el método del codo para K de 2 a {K_MAX - 1}...")
for k in range(2, K_MAX):
    modelo_k = KMeans(featuresCol="features_scaled", k=k, seed=42).fit(df_final)
    cost.append(modelo_k.summary.trainingCost)
    print(f"  K={k}  costo={cost[-1]:,.0f}")

In [ ]:
df_elbow = pd.DataFrame({'K': list(range(2, K_MAX)), 'WSS': cost})
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_elbow['K'], df_elbow['WSS'], marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=6)
ax.set_title('Método del Codo (Selección de K)', fontsize=14, pad=15, fontweight='bold')
ax.set_xlabel('K'); ax.set_ylabel('WSS (costo de entrenamiento)')
ax.grid(True, linestyle='--', alpha=0.5, color='#ccc')
ax.set_xticks(df_elbow['K'])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.show()

In [ ]:
%%time
k_optimo = 4
kmeans_final = KMeans(featuresCol="features_scaled", k=k_optimo, predictionCol="cluster", seed=42)
model_final = kmeans_final.fit(df_final)
df_res = model_final.transform(df_final)

silueta = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster",
                              metricName="silhouette").evaluate(df_res.sample(0.05, seed=42))
print(f"Silueta (muestra 5%) con K={k_optimo}: {silueta:.3f}")

print("\nLavado por cluster (¿se concentra en alguno?):")
(df_res.groupBy("cluster")
       .agg(F.count("*").alias("transacciones"),
            F.round(F.avg("label_real") * 100, 4).alias("pct_lavado"))
       .orderBy("cluster").show())

# BLOQUE B — Clasificación (Random Forest)

`laundering_fraction` en `configs/gcp/aml-pilot.json` es 0.004 — menos extremo que el
~0.10% de IBM, pero igual de necesario ponderar las clases para que el modelo no
converja a "siempre normal".

In [ ]:
%%time
n = df_final.count()
n_pos = df_final.filter(col("label_real") == 1).count()
n_neg = n - n_pos
print(f"Lavado: {n_pos:,} de {n:,} ({n_pos / n:.4%}). Un modelo que siempre diga 'normal' acierta {n_neg / n:.4%}.")

df_clf = df_final.withColumn("peso", F.when(col("label_real") == 1, n / (2.0 * n_pos)).otherwise(n / (2.0 * n_neg)))
train, test = df_clf.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestClassifier(labelCol="label_real", featuresCol="features_raw", weightCol="peso",
                            numTrees=50, seed=42)
model_rf = rf.fit(train)
preds = model_rf.transform(test)

auc_roc = BinaryClassificationEvaluator(labelCol="label_real", metricName="areaUnderROC").evaluate(preds)
auc_pr  = BinaryClassificationEvaluator(labelCol="label_real", metricName="areaUnderPR").evaluate(preds)
print(f"\n>>> AUC-ROC = {auc_roc:.4f} | AUC-PR = {auc_pr:.4f} (azar: AUC-ROC 0.5, AUC-PR ≈ {n_pos / n:.4f}) <<<")

# BLOQUE C — Detección de anomalías por error de reconstrucción

Mismo autoencoder lineal (PCA, ajustado solo con transacciones normales de train) que en
los otros dos notebooks. Split antes de ajustar, el "encoder" solo ve normales de train,
umbral definido con datos de train.

In [ ]:
%%time
train_c, test_c = df_base.randomSplit([0.8, 0.2], seed=42)

scaler_c = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
prep_c = Pipeline(stages=indexers + [encoder, assembler, scaler_c]).fit(train_c)
train_f = prep_c.transform(train_c)
test_f = prep_c.transform(test_c)

K_BOTTLENECK = 6
normales_train = train_f.filter(col("label_real") == 0)
ae = PCA(k=K_BOTTLENECK, inputCol="features_scaled", outputCol="codigo").fit(normales_train)
print(f"Variables de entrada: {len(train_f.select('features_scaled').first()[0])} -> cuello de botella: {K_BOTTLENECK}")
print(f"Información que conserva el cuello de botella: {sum(ae.explainedVariance):.1%}")

def norma2(c):
    return F.aggregate(vector_to_array(c), F.lit(0.0), lambda acc, x: acc + x * x)

def puntuar(d):
    d = ae.transform(d)
    return d.withColumn("anomaly_score", F.sqrt(F.greatest(norma2("features_scaled") - norma2("codigo"), F.lit(0.0))))

train_scored = puntuar(train_f)
df_predictions = puntuar(test_f).persist()

In [ ]:
umbral = train_scored.filter(col("label_real") == 0).approxQuantile("anomaly_score", [0.95], 0.001)[0]
print(f"Umbral (percentil 95 de normales en train): {umbral:.4f}")

df_predictions = df_predictions.withColumn("alerta", (col("anomaly_score") > umbral).cast("int"))
print("\nMatriz de confusión (filas = realidad, columnas = alerta):")
df_predictions.groupBy("label_real").pivot("alerta", [0, 1]).count().na.fill(0).orderBy("label_real").show()

cm = {(r["label_real"], r["alerta"]): r["count"] for r in df_predictions.groupBy("label_real", "alerta").count().collect()}
tp, fn, fp = cm.get((1.0, 1), 0), cm.get((1.0, 0), 0), cm.get((0.0, 1), 0)
print(f"Lavado detectado (recall): {tp / max(tp + fn, 1):.1%} | de las alertas, lavado real (precisión): {tp / max(tp + fp, 1):.1%}")

roc_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderROC").evaluate(df_predictions)
pr_ae = BinaryClassificationEvaluator(labelCol="label_real", rawPredictionCol="anomaly_score", metricName="areaUnderPR").evaluate(df_predictions)
print(f"AUC-ROC = {roc_ae:.4f} | AUC-PR = {pr_ae:.4f}")

## Para practicar

1. **Split por shard, no por fila.** Repite el Bloque B usando `shard_id` para separar
   train/test (todos los shards pares a train, impares a test) en vez de `randomSplit`
   aleatorio — ¿cambia mucho el AUC-PR? Es la prueba real de que el modelo generaliza
   entre poblaciones independientes, no solo dentro de una.
2. **Motivo, no solo label:** entrena un clasificador **multiclase** sobre
   `scenario_motif` (`fan_in`/`fan_out`/`chain`/`cycle`/`none`) en vez de binario sobre
   `is_laundering` — ¿el modelo distingue mejor la topología que el intento?
3. **`summaries/account_summaries.parquet`:** ya trae features de cuenta para todo el
   periodo (no solo "antes del evento" como `sender_prior_*`) — apto para clustering de
   cuentas, **no** para predecir una transacción individual sin fuga (mezclaría futuro).
4. **Comparación de tres vías:** corre este notebook, `Fraud_Detection.ipynb` y
   `Fraud_Detection_AML_HF.ipynb` con las mismas métricas y compara qué tan distinto es
   detectar fraude en datos sintéticos de curso, en datos sintéticos propios con
   topología documentada, y en un benchmark de investigación real.

In [ ]:
spark.stop()